# A capacity-limited MoE layer in NumPy

A top-2 MoE layer whose experts have fixed-size buffers does four things to a batch: it routes
each token, decides which assignments fit in the buffers, runs every expert on the assignments it
kept, and adds the weighted outputs to the residual stream. You will write those steps for a batch
of 64 tokens and count the tokens that the buffers shut out.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np


def softmax(z):
    """Probabilities along the last axis, after subtracting each row's maximum."""
    shifted = z - np.max(z, axis=-1, keepdims=True)
    weights = np.exp(shifted)
    return weights / np.sum(weights, axis=-1, keepdims=True)


# 64 tokens of width 8 and four experts, each a small ReLU network 8 -> 16 -> 8.
# The router favors experts 0 and 1.
rng = np.random.default_rng(50)
T, d, h, N, k = 64, 8, 16, 4, 2
X = rng.normal(size=(T, d)) + 0.5
Wg = rng.normal(size=(N, d)) * 0.5
Wg[0] += 0.3
Wg[1] += 0.2
W1 = rng.normal(size=(N, d, h)) / np.sqrt(d)
W2 = rng.normal(size=(N, h, d)) / np.sqrt(h)


def expert(e, x):
    """Expert e applied to every row of x."""
    return np.maximum(x @ W1[e], 0) @ W2[e]


print(f"{T} tokens of width {d}, {N} experts, top-{k} routing")

## Part 1 (coding task): Top-k gates

Write `top_k_gates(logits, k)` for router logits of shape $(T, N)$. It returns two arrays of shape
$(T, k)$:

- `idx`: each token's $k$ experts with the highest router probability, the most probable first.
- `gates`: their gate weights, renormalized over the $k$ chosen experts as in Mixtral.

In [ ]:
def top_k_gates(logits, k):
    """Each token's top-k experts, most probable first, and their renormalized gates."""
    # TODO
    raise NotImplementedError

In [ ]:
logits = np.array([[0.0, 2.0, 1.0, -1.0],
                   [1.5, 0.0, 0.5, 3.0],
                   [0.2, 0.1, 0.4, 0.3]])
idx, gates = top_k_gates(logits, 2)
idx, gates = np.asarray(idx), np.asarray(gates, dtype=float)
assert idx.shape == (3, 2) and gates.shape == (3, 2), f"expected two arrays of shape (3, 2), got {idx.shape} and {gates.shape}"
assert np.array_equal(np.sort(idx, axis=1), [[1, 2], [0, 3], [2, 3]]), f"wrong experts chosen: {idx.tolist()}"
assert np.array_equal(idx[:, 0], [1, 3, 2]), f"column 0 should hold each token's most probable expert, got {idx[:, 0].tolist()}"
assert np.allclose(gates.sum(axis=1), 1), f"each token's gates should add to 1, got {np.round(gates.sum(axis=1), 4).tolist()}"
assert np.allclose(gates[:, 0], [0.7310586, 0.8175745, 0.5249792]), f"wrong gate values: {np.round(gates, 4).tolist()}"
print("Part 1 passed")

## Part 2 (coding task): Which assignments fit

Each expert has a buffer of `capacity` slots and fills it in this order: every token's first
choice, from token 0 to token $T - 1$, then every token's second choice in the same token order,
and so on up to the $k$-th choice. An assignment is kept when its expert has accepted fewer than
`capacity` assignments before it, and dropped otherwise.

Write `dispatch(idx, n_experts, capacity)`. It returns a boolean array `keep` with the shape of
`idx`, true for the assignments that are kept.

In [ ]:
def dispatch(idx, n_experts, capacity):
    """Boolean array shaped like idx: which assignments fit in their expert's buffer."""
    # TODO
    raise NotImplementedError

In [ ]:
small = np.array([[0, 1], [1, 0], [0, 2], [0, 1], [2, 0]])
keep = np.asarray(dispatch(small, 3, 2))
assert keep.shape == small.shape and keep.dtype == bool, "keep should be a boolean array shaped like idx"
assert not (keep[1, 1] and not keep[2, 0]), (
    "token 2's first choice was dropped while token 1's second choice was kept: "
    "every first choice is served before any second choice"
)
assert np.array_equal(keep, [[True, True], [True, False], [True, True], [False, False], [True, False]]), (
    f"wrong assignments kept: {keep.tolist()}"
)
idx_all, _ = top_k_gates(X @ Wg.T, k)
keep_all = np.asarray(dispatch(idx_all, N, 25))
for e in range(N):
    assert keep_all[idx_all == e].sum() <= 25, f"expert {e} accepted more than its 25 slots"
assert keep_all.sum() == 71, f"expected 71 of the 128 assignments to fit 25 slots per expert, got {keep_all.sum()}"
print("Part 2 passed")

## Part 3 (coding task): The layer

Write `moe_layer(X, Wg, k, cf)`. It routes the tokens with `top_k_gates(X @ Wg.T, k)`, gives every
expert the capacity from the lesson's formula for capacity factor `cf`, rounded down to a whole
number of slots, keeps the assignments that `dispatch` keeps, and returns

- `Y`: each token's input plus the gate-weighted outputs of its kept experts, computed with
  `expert(e, x)` from the setup. A dropped assignment adds nothing, and the token's other gate
  keeps its value.
- `skipped`: the number of tokens whose assignments were all dropped.

Run each expert once, on all the tokens it kept.

In [ ]:
def moe_layer(X, Wg, k, cf):
    """Output of the MoE layer with its residual, and the number of tokens that skip it."""
    T, N = len(X), len(Wg)
    # TODO
    raise NotImplementedError

In [ ]:
Y, skipped = moe_layer(X, Wg, k, 1.0)
Y = np.asarray(Y, dtype=float)
assert Y.shape == X.shape, f"Y should have the shape of X, {X.shape}, got {Y.shape}"
assert skipped != 18, "18 tokens skip at capacity factor 1.0: every buffer is half the size it should be"
assert skipped == 3, f"expected 3 tokens to skip the layer at capacity factor 1.0, got {skipped}"
assert np.allclose(Y[[59, 60, 62]], X[[59, 60, 62]]), "tokens 59, 60 and 62 lose every assignment, so they pass through unchanged"
assert np.isclose(Y.sum(), 379.3523553, atol=1e-5) and np.isclose((Y * Y).sum(), 1048.6355384, atol=1e-5), (
    "Y is off: add each kept expert's output, times its own gate, to the token's input"
)
Y_free, skipped_free = moe_layer(X, Wg, k, 4.0)
assert skipped_free == 0 and np.isclose(np.asarray(Y_free).sum(), 417.2081216, atol=1e-5), "wrong output when no assignment is dropped"
for cf in (1.0, 1.25, 1.5, 2.0):
    print(f"capacity factor {cf:<4}  tokens that skip the layer: {moe_layer(X, Wg, k, cf)[1]}")
print("Part 3 passed")

## Solution

### Part 1: Top-k gates

In [ ]:
def top_k_gates(logits, k):
    """Each token's top-k experts, most probable first, and their renormalized gates."""
    probs = softmax(logits)
    idx = np.argsort(-probs, axis=1)[:, :k]
    chosen = np.take_along_axis(probs, idx, axis=1)
    return idx, chosen / chosen.sum(axis=1, keepdims=True)

**Why it works.** The softmax keeps the order of the logits, so the top $k$ probabilities belong to the top $k$
logits, and sorting by `-probs` puts the most probable first. Dividing the chosen probabilities by
their sum cancels the softmax denominator: the gates are a softmax over the $k$ chosen logits
alone. For $k = 2$ that is $\sigma(z_a - z_b)$, which is why the first gates in the check are
$\sigma(1) = 0.731$, $\sigma(1.5) = 0.818$ and $\sigma(0.1) = 0.525$.

### Part 2: Which assignments fit

In [ ]:
def dispatch(idx, n_experts, capacity):
    """Boolean array shaped like idx: which assignments fit in their expert's buffer."""
    T, k = idx.shape
    keep = np.zeros((T, k), dtype=bool)
    used = np.zeros(n_experts, dtype=int)
    for r in range(k):                   # every first choice, then every second choice, ...
        for t in range(T):               # ... each in token order
            e = idx[t, r]
            if used[e] < capacity:
                keep[t, r] = True
                used[e] += 1
    return keep

**Why it works.** The outer loop runs over the choice rank and the inner one over the tokens, so a token's second
choice never takes a slot that some later token's first choice needed. The order matters most for
the tokens that skip the layer. On this batch at capacity factor 1.25, the layer drops 27 of the
128 assignments, all of them second choices, and every token keeps its first expert. Filling the
buffers token by token, both choices of token 0 before any choice of token 1, also drops 27
assignments, but 11 tokens lose both of theirs and skip the layer. The loop over
$T k$ assignments is fine at this size. A vectorized version numbers each assignment's arrival
at its expert with a cumulative count over the rank-major order and keeps those numbered below
the capacity.

### Part 3: The layer

In [ ]:
def moe_layer(X, Wg, k, cf):
    """Output of the MoE layer with its residual, and the number of tokens that skip it."""
    T, N = len(X), len(Wg)
    idx, gates = top_k_gates(X @ Wg.T, k)
    capacity = int(cf * k * T / N)
    keep = dispatch(idx, N, capacity)
    Y = X.copy()
    for e in range(N):
        rows, ranks = np.nonzero((idx == e) & keep)    # the tokens expert e kept, and in which column
        Y[rows] += gates[rows, ranks, None] * expert(e, X[rows])
    skipped = int((~keep).all(axis=1).sum())
    return Y, skipped

**Why it works.** A token makes $k$ assignments, so a perfectly balanced expert receives $kT/N = 32$ of them, and
capacity factor 1.0 gives 32 slots. Switch's formula without the $k$ would give 16 slots, and 18
tokens would skip the layer instead of 3. The router sends 53 and 54 assignments to experts 0 and
1 and only 10 and 11 to the other two, so at factor 1.0 those two experts drop 43 assignments, and
tokens 59, 60 and 62 lose both of theirs. Because a dropped assignment's gate is not handed to the
other expert, a token that loses one assignment gets a smaller update than usual, not a different
mix: its output moves toward its input. Each expert runs once on a gathered block of tokens, the
way a real implementation fills a buffer and runs one matrix product per expert.